In [16]:
import numpy as np
import pandas as pd
from scipy import stats

In [17]:
# Load the cleaned dataset
df = pd.read_csv("../data/processed/marketing_cleaned.csv")

In [18]:
# Extract experimental group totals
n_ad = (df['test_group'] == 'ad').sum()
x_ad = df.loc[df['test_group'] == 'ad', 'converted'].sum()

n_psa = (df['test_group'] == 'psa').sum()
x_psa = df.loc[df['test_group'] == 'psa', 'converted'].sum()

p_ad = x_ad / n_ad
p_psa = x_psa / n_psa

In [19]:
# Absolute Difference & Relative Lift
diff = p_ad - p_psa
relative_lift = (diff / p_psa) * 100

In [20]:
# Two-Proportion Pooled Z-Test
# Under H0, both groups share a single pooled conversion rate:
p_pooled = (x_ad + x_psa) / (n_ad + n_psa)
se_pooled = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_ad + 1 / n_psa))
z_stat = diff / se_pooled
p_value_z = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# Chi-Square Test of Independence (2x2 Contingency Table)
contingency_table = np.array([
    [x_ad, n_ad - x_ad],
    [x_psa, n_psa - x_psa]
])
chi2_stat, p_value_chi2, dof, _ = stats.chi2_contingency(contingency_table, correction=False)

# 95% Confidence Interval for Absolute Risk Difference (Unpooled Standard Error)
se_unpooled = np.sqrt((p_ad * (1 - p_ad) / n_ad) + (p_psa * (1 - p_psa) / n_psa))
z_crit = stats.norm.ppf(0.975)  # 1.95996 for 95% two-sided CI
ci_lower = diff - z_crit * se_unpooled
ci_upper = diff + z_crit * se_unpooled

# Print Formatted Reporting
print("=================== STATISTICAL HYPOTHESIS TEST ===================")
print(f"Ad Cohort:   n = {n_ad:,} | Conversions = {x_ad:,} | CVR = {p_ad:.6f} ({p_ad*100:.4f}%)")
print(f"PSA Cohort:  n = {n_psa:,} | Conversions = {x_psa:,}  | CVR = {p_psa:.6f} ({p_psa*100:.4f}%)")
print("-" * 65)
print(f"Absolute Lift (Delta):  {diff:+.6f} ({diff*100:+.4f} percentage points)")
print(f"Relative Lift:         {relative_lift:+.2f}%")
print(f"Standard Error (Diff):  {se_unpooled:.6f}")
print("-" * 65)
print(f"Two-Proportion Z-Statistic: {z_stat:.4f}")
print(f"Two-Proportion p-value:     {p_value_z:.6e}")
print(f"Chi-Square Statistic:       {chi2_stat:.4f} (p-value: {p_value_chi2:.6e})")
print(f"95% Confidence Interval:    [{ci_lower:+.6f}, {ci_upper:+.6f}]")
print(f"95% CI (Percentage Points): [{ci_lower*100:+.4f}%, {ci_upper*100:+.4f}%]")

# Check Statistical Assumptions
print("\n=================== ASSUMPTIONS AUDIT ===================")
print(f"Normality / Large Sample Condition (np >= 10 and n(1-p) >= 10):")
print(f"  Ad Group:  Successes = {x_ad:,} | Failures = {n_ad - x_ad:,}")
print(f"  PSA Group: Successes = {x_psa:,}  | Failures = {n_psa - x_psa:,}")
print(f"  All counts >= 10: {all(count >= 10 for count in [x_ad, n_ad - x_ad, x_psa, n_psa - x_psa])}")

=================== STATISTICAL HYPOTHESIS TEST ===================
Ad Cohort:   n = 564,577 | Conversions = 14,423 | CVR = 0.025547 (2.5547%)
PSA Cohort:  n = 23,524 | Conversions = 420  | CVR = 0.017854 (1.7854%)
-----------------------------------------------------------------
Absolute Lift (Delta):  +0.007692 (+0.7692 percentage points)
Relative Lift:         +43.09%
Standard Error (Diff):  0.000889
-----------------------------------------------------------------
Two-Proportion Z-Statistic: 7.3701
Two-Proportion p-value:     1.705303e-13
Chi-Square Statistic:       54.3181 (p-value: 1.705281e-13)
95% Confidence Interval:    [+0.005951, +0.009434]
95% CI (Percentage Points): [+0.5951%, +0.9434%]

=================== ASSUMPTIONS AUDIT ===================
Normality / Large Sample Condition (np >= 10 and n(1-p) >= 10):
  Ad Group:  Successes = 14,423 | Failures = 550,154
  PSA Group: Successes = 420  | Failures = 23,104
  All counts >= 10: True


Business Impact & Sensitivity Modeling

In [26]:
# 1. Incremental Volume Sizing
incremental_conversions = n_ad * diff
incremental_lower = n_ad * ci_lower
incremental_upper = n_ad * ci_upper

print("================ BUSINESS IMPACT SIZING ================")
print(f"Total Ad Cohort Users: {n_ad:,}")
print(f"Observed Conversions:  {x_ad:,}")
print(f"Baseline Expected Conversions (at PSA rate 1.785%): {int(round(n_ad * p_psa)):,}")
print(f"Estimated Incremental Conversions: {int(round(incremental_conversions)):,} "
      f"(95% CI: [{int(round(incremental_lower)):,}, {int(round(incremental_upper)):,}])")
print(f"Lift per 1,000 Users Reached: {diff * 1000:.2f} additional customers")

================ BUSINESS IMPACT SIZING ================
Total Ad Cohort Users: 564,577
Observed Conversions:  14,423
Baseline Expected Conversions (at PSA rate 1.785%): 10,080
Estimated Incremental Conversions: 4,343 (95% CI: [3,360, 5,326])
Lift per 1,000 Users Reached: 7.69 additional customers


In [25]:
# 2. Financial Sensitivity Scenario (Break-Even Unit Economics)
# Average ads seen per user in ad group:
avg_ads_per_user = df.loc[df['test_group'] == 'ad', 'total_ads'].mean()
total_impressions_ad = df.loc[df['test_group'] == 'ad', 'total_ads'].sum()

print("\n================ AD SPEND SENSITIVITY ================")
print(f"Total Ad Impressions Served: {total_impressions_ad:,}")
print(f"Average Impressions per User: {avg_ads_per_user:.2f}")

# Evaluate breakeven gross margin per conversion under different CPM scenarios
cpm_scenarios = [2.00, 5.00, 10.00, 15.00]  # Cost per 1,000 impressions

print("\nBreak-Even Gross Margin Required per Incremental Conversion:")
for cpm in cpm_scenarios:
    total_cost = (total_impressions_ad / 1000) * cpm
    cost_per_incremental_conv = total_cost / incremental_conversions
    cost_per_user = total_cost / n_ad
    print(f"  At CPM = ${cpm:5.2f} | Total Spend: ${total_cost:11,.2f} | "
          f"Break-Even Margin Required: ${cost_per_incremental_conv:6.2f} per conversion")


================ AD SPEND SENSITIVITY ================
Total Ad Impressions Served: 14,014,701
Average Impressions per User: 24.82

Break-Even Gross Margin Required per Incremental Conversion:
  At CPM = $ 2.00 | Total Spend: $  28,029.40 | Break-Even Margin Required: $  6.45 per conversion
  At CPM = $ 5.00 | Total Spend: $  70,073.50 | Break-Even Margin Required: $ 16.13 per conversion
  At CPM = $10.00 | Total Spend: $ 140,147.01 | Break-Even Margin Required: $ 32.27 per conversion
  At CPM = $15.00 | Total Spend: $ 210,220.51 | Break-Even Margin Required: $ 48.40 per conversion
